# Shape representation

In this practical session, we investigate the possibility to representation for implicit functions.

## Get the data

In [ ]:
import os
if not os.path.exists("data"):
  if not os.path.exists("msia_shapenet_bench.zip"):
      !wget https://github.com/aboulch/MSIA_points/releases/download/v0.0.0/msia_shapenet_bench.zip
  if not os.path.exists("msia_subset"):
    !unzip -qq msia_shapenet_bench
  !mkdir data
  !mv msia_subset data/

## Imports for the session

In [ ]:
from sklearn.metrics import confusion_matrix
import plotly.graph_objects as go # for visualization
import tqdm
import os
import h5py
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from scipy.spatial import KDTree
import matplotlib.pyplot as plt
import json
from sklearn.metrics import confusion_matrix
import glob

## Visualization function

In [ ]:
# display the point cloud
def point_cloud_visu(pts, cls=None):

    fig = go.Figure(
        data=[
            go.Scatter3d(
                x=pts[:,0], y=pts[:,1], z=pts[:,2],
                mode='markers',
                marker=dict(size=3,
                            color=cls,
                            colorscale='Viridis',
                            )
            )
        ],
        layout=dict(
            scene=dict(
                xaxis=dict(visible=False),
                yaxis=dict(visible=False),
                zaxis=dict(visible=False),
                aspectmode="data", #this string can be 'data', 'cube', 'auto', 'manual'
                #a custom aspectratio is defined as follows:
                aspectratio=dict(x=1, y=1, z=0.95)
            )
        )
    )
    fig.show()

## Overfitting one shape

The objective is to learn the occupany of one shape and being able to decode it. Here we will use a decoder-only approach, the occupancy function will be stored in the network weights.

### Loading a point cloud

The data folder contains `.npz` files, one file contains occupancy points in the space with corresponding labels (0 for outside, 1 for inside).
Each file represent one shape (of the class Bench in ShapeNet).

**Question:** Create the function to load the points with there occupancy. The function should return a point cloud ($N \times 3$ tensor) and the occupancy (a tensor containing integers of size $N$).

The occupancies are compressed, to get them use the `np.unpackbits` function.

In [ ]:

def get_data_occupancy(folder_path):

    points = np.load(os.path.join(folder_path, "points.npz"))
    points_space = torch.tensor(points["points"], dtype=torch.float)
    occupancies = torch.tensor(np.unpackbits(points['occupancies']), dtype=torch.long)
    points_space = points_space[:,[0,2,1]]
    return points_space, occupancies

folder_path = "msia_subset/10654ea604644c8eca7ed590d69b9804"
pts, occ = get_data_occupancy(folder_path)
mask = (occ==1)
point_cloud_visu(pts[mask], pts[mask][:,2])

### Network definition

The network used here has a decoder only structure.
It will contains 2 elements:
* an embedding (`torch.nn.Embedding`) which will contain the shape embeddings of size `embedding_dim`
* an MLP (e.g., 5 linear layers separated by GELU layers)
  * the input size is `embedding_dim+3`, the 3 being for the query point (the location in the space where we want to konw the occupancy)
  * the output size is 1, the occupancy

**Question:** create the network class.

In [ ]:
class SDFNetwork(torch.nn.Module):

    def __init__(self, num_embeddings, embedding_dim):
        super().__init__()

        self.shape_embedding = torch.nn.Embedding(num_embeddings, embedding_dim)

        self.decoder = torch.nn.Sequential(
            torch.nn.Linear(embedding_dim+3, 512),
            torch.nn.GELU(),
            torch.nn.Linear(512, 512),
            torch.nn.GELU(),
            torch.nn.Linear(512, 512),
            torch.nn.GELU(),
            torch.nn.Linear(512, 512),
            torch.nn.GELU(),
            torch.nn.Linear(512, 1),
        )

    def forward(self, shape_id, queries):

        # 1 call the shape embedding with shape_id to get the correct embedding
        shape_embeddings = self.shape_embedding(shape_id)
        shape_embeddings = shape_embeddings.unsqueeze(1).repeat(1, queries.shape[1], 1)

        # 2 concatenate the embedding with the queries
        x = torch.cat([shape_embeddings, queries], dim=-1)

        # 3 forward pass to the MLP
        x = self.decoder(x)

        return x.squeeze(-1) # squeeze last dimension (return size BxN)

### Parameters

We give here parameters for the training:
* the folder path to learn the occupancy of a single shape
* the points and there occupancy
* the number of iterations for training
* the maximal number of points use in one batch
* the device
* the latent size

In [ ]:
folder_path = "data/msia_subset/10654ea604644c8eca7ed590d69b9804"
pts, occ = get_data_occupancy(folder_path)
num_iterations = 30000
num_points_occ = 2000 # upper bound for the points
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
latent_size = 512

As the there are much more points with emty occupancy.
We will balance the training by randomly selecting the same amount of points with 0 label and 1 label.

**Question**: fill the script to train a model (and save the weights in `model.ckpt`, it can be saved every 1000 iterations).

In [ ]:
# create the network and move it to the right device
# ...
network = SDFNetwork(1, latent_size)
network.to(device)

# create the optimizer (Adam or AdamW)
# ...
optimizer = torch.optim.Adam(network.parameters(), lr=0.0001)

epoch_step = 0
epoch_loss = 0
epoch_acc = 0

# get the points with 0/1 occupancy ()
# occ_pts_1 = ...
# occ_pts_0 = ...
mask = (occ == 1)
occ_pts_1 = pts[mask]
occ_1 = occ[mask]
occ_pts_0 = pts[~mask]
occ_0 = occ[~mask]

# main loop
# - set the model to train mode
# - iterate
# - select the same amount of points with label 1 and 0 (max is num_points_occ)
# - predict the occupancy
# - use a binary cross entropy loss (with logits)
# - save the model every 1000 iterations
# - (optional) print with the loss and the prediction accuracy

network.train()
t = tqdm.tqdm(range(num_iterations), ncols=100)
for iter_id in t:

    if occ_pts_1.shape[0] < num_points_occ//2:
        num_points_occ_tmp = occ_pts_1.shape[0]
    else:
        num_points_occ_tmp = num_points_occ//2

    ids_0 = torch.randperm(occ_pts_0.shape[0], dtype=torch.long)[:num_points_occ_tmp]
    ids_1 = torch.randperm(occ_pts_1.shape[0], dtype=torch.long)[:num_points_occ_tmp]

    occ_pts_0_rand = occ_pts_0[ids_0].unsqueeze(0).to(device)
    occ_0_rand = occ_0[ids_0].unsqueeze(0).to(device).float()

    occ_pts_1_rand = occ_pts_1[ids_1].unsqueeze(0).to(device)
    occ_1_rand = occ_1[ids_1].unsqueeze(0).to(device).float()

    occ_pts_rand = torch.cat([occ_pts_0_rand, occ_pts_1_rand], dim=1)
    occ_rand = torch.cat([occ_0_rand, occ_1_rand], dim=1).float()


    shape_ids = torch.tensor([0], dtype=torch.long, device=device)

    occ_pred = network(shape_ids, occ_pts_rand)

    loss = torch.nn.functional.binary_cross_entropy_with_logits(occ_pred, occ_rand.float())

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    epoch_loss += loss.item()
    epoch_step += 1

    occ_pred_labels = (torch.sigmoid(occ_pred) > 0.5).long()
    acc = (occ_pred_labels == occ_rand).float().mean()
    epoch_acc += acc.item()



    t.set_description_str(f"Loss: {epoch_loss/epoch_step:.4f} - Acc: {epoch_acc/epoch_step:.4f}")

    if iter_id % 100 == 0:
        global_step = 0
        global_loss = 0
        global_acc = 0

    if iter_id % 1000 == 0:
        torch.save(network.state_dict(), "model.ckpt")

### Extraction of the mesh

Now we want the extract the mesh for the learned shape.
To do so, we will use the marching cubes algorithm (https://en.wikipedia.org/wiki/Marching_cubes) from scikit image.

**Question**: fill the `get_dense_pointcloud` that compute a point cloud for a regular grid of size $[s \times s \times s]$ where s in the number of steps (number of points following one direction). One can use the `meshgrid` function from pytorch. The coordinates of the points should range in $[-0.5,0.5]^3$

**Question** Compute the occupancy using the `model.ckpt` previously saved and compute the marching cubes.

We provide the visualization function.

In [ ]:
from skimage import measure

num_steps = 32

def get_dense_pointcloud(num_steps):
    # compute the point coordinates for a regular grid
    # ...
    x = torch.arange(-0.5,0.5,1/num_steps)
    y = torch.arange(-0.5,0.5,1/num_steps)
    z = torch.arange(-0.5,0.5,1/num_steps)
    vox_coords = torch.stack(torch.meshgrid(x, y, z, indexing='ij'), dim=-1)
    return vox_coords

# get the query points
query_points = get_dense_pointcloud(num_steps).unsqueeze(0).to(device)
query_points = query_points.reshape(1,-1,3)

# inference through the model
# ...
network = SDFNetwork(1, latent_size)
network.load_state_dict(torch.load("model.ckpt"))
network.to(device)
network.eval()

with torch.no_grad():
  shape_ids = torch.tensor([0], dtype=torch.long, device=device)
  occ_pred = network(shape_ids, query_points).squeeze(0)
  occ_pred = torch.sigmoid(occ_pred)
  occ_pred = occ_pred.reshape(num_steps, num_steps, num_steps)

# marching cubes
#....
occ_pred = occ_pred.cpu().numpy()
verts, faces, normals, values = measure.marching_cubes(occ_pred, 0.5)
verts = verts/num_steps - 0.5

# visualization
fig = go.Figure(go.Mesh3d(
        x=verts[:,0], y=verts[:,1], z=verts[:,2],
        i=faces[:,0], j=faces[:,1], k=faces[:,2],))
fig.show()

## Training with multiple shape

Overfitting a single shape is interesting but limited.
Here we train a single decoder for many shapes.
The idea is that the embeddings will contain the shape information and the network will only predict the occupancy.

**Question:** create the dataset to load the shapes.
* the init function load the `train.lst` which contains the training shapes name
* the getitem function load the data and returns:
  * the points
  * the occupancy
  * the id of the shape to access the corect slot in the network embedding

In [ ]:
class ShapeNet(torch.utils.data.Dataset):

    def __init__(self, root, split="training",
                     filter_name=None,
                     num_non_manifold_points=2048,
                     dataset_size=None, **kwargs):

        super().__init__()

        self.root = root
        self.split = split
        self.filter_name = filter_name
        self.filelists = []
        self.num_non_manifold_points = num_non_manifold_points

        # get the lists for files for the split:
        # ...

        if split in ["train", "training"]:
            for path in glob.glob(os.path.join(self.root,"*/train.lst")):
                self.filelists.append(path)
        elif split in ["validation", "val"]:
            for path in glob.glob(os.path.join(self.root,"*/val.lst")):
                self.filelists.append(path)
        elif split in ["trainVal", "trainingValidation", "training_validation"]:
            for path in glob.glob(os.path.join(self.root,"*/train.lst")):
                self.filelists.append(path)
            for path in glob.glob(os.path.join(self.root,"*/val.lst")):
                self.filelists.append(path)
        elif split in ["test", "testing"]:
            for path in glob.glob(os.path.join(self.root,"*/test.lst")):
                self.filelists.append(path)

        # sort the file list such that it has always the same order
        self.filelists.sort()

        # compute the paths
        self.filenames = []
        for flist in self.filelists:
            with open(flist) as f:
                dirname = os.path.dirname(flist)
                content = f.readlines()
                content = [line.split("\n")[0] for line in content]
                content = [os.path.join(dirname, line) for line in content]
            if dataset_size is not None:
                content = content[:dataset_size]
            self.filenames += content

        # check that paths are valid
        fnames = []
        for fname in self.filenames:
            if os.path.exists(fname):
                fnames.append(fname)
        self.filenames = fnames


    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):
        """Get item."""
        folder_path = self.filenames[idx]

        # get the points
        occ_pts, occ = get_data_occupancy(folder_path)

        # sample the same number of points with 0/1 label
        # ...
        mask = (occ == 1)
        occ_pts_1 = occ_pts[mask]
        occ_1 = occ[mask]
        occ_pts_0 = occ_pts[~mask]
        occ_0 = occ[~mask]

        ids_1 = torch.randperm(occ_pts_1.shape[0], dtype=torch.long)[:self.num_non_manifold_points//2]

        num_pts_0 = self.num_non_manifold_points - ids_1.shape[0]
        ids_0 = torch.randperm(occ_pts_0.shape[0], dtype=torch.long)[:num_pts_0]

        occ_pts_0_rand = occ_pts_0[ids_0]
        occ_0_rand = occ_0[ids_0].float()

        occ_pts_1_rand = occ_pts_1[ids_1]
        occ_1_rand = occ_1[ids_1].float()

        occ_pts_rand = torch.cat([occ_pts_0_rand, occ_pts_1_rand], dim=0)
        occ_rand = torch.cat([occ_0_rand, occ_1_rand], dim=0).float()

        # return the id of the shape, the points and the occupancy
        return idx, occ_pts_rand, occ_rand

### Training

As before we provide some parameters for the network

In [ ]:

folder_path = "data/"
num_iterations = 30000
num_points_occ = 10000 # upper bound for the points
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
latent_size = 512
num_workers = 2
batch_size = 16
num_epochs = 100

**Question:** train the network:
* the dataset / dataloader
* the optimizer
* loop for a given number of epochs
* save the network after each epoch (name `model_multi.ckpt`)

In [ ]:

dataset_train = ShapeNet(folder_path, split="train")
dataloader_train = torch.utils.data.DataLoader(dataset_train, batch_size=batch_size, shuffle=True, num_workers=num_workers)

print(len(dataset_train))

# create the network
network = SDFNetwork(len(dataset_train), latent_size)
network.to(device)

# create the optimizer
optimizer = torch.optim.Adam(network.parameters(), lr=0.0001)

for epoch in range(num_epochs):

    epoch_step = 0
    epoch_loss = 0
    epoch_acc = 0

    t = tqdm.tqdm(dataloader_train, ncols=100)
    for data in t:

        shape_ids = data[0].long().to(device)
        occ_pts = data[1].float().to(device)
        occ = data[2].float().to(device)


        occ_pred = network(shape_ids, occ_pts)

        loss = torch.nn.functional.binary_cross_entropy_with_logits(occ_pred, occ)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        epoch_step += 1

        occ_pred_labels = (torch.sigmoid(occ_pred) > 0.5).long()
        acc = (occ_pred_labels == occ).float().mean()
        epoch_acc += acc.item()

        t.set_description_str(f"Loss: {epoch_loss/epoch_step:.4f} - Acc: {epoch_acc/epoch_step:.4f}")

    torch.save(network.state_dict(), "model_multi.ckpt")



### Shape prediction

**Question:** predict and display the mesh for a particular shape id in the embedding

For faster iteration, you can use a pretrained model (1000 epochs).

In [ ]:
if not os.path.exists("msia_8_model_multi.ckpt"):
  ! wget https://github.com/aboulch/MSIA_points/releases/download/v0.0.0/msia_8_model_multi.ckpt

from skimage import measure

num_steps = 32

def get_dense_pointcloud(num_steps):
    x = torch.arange(-0.5,0.5,1/num_steps)
    y = torch.arange(-0.5,0.5,1/num_steps)
    z = torch.arange(-0.5,0.5,1/num_steps)
    vox_coords = torch.stack(torch.meshgrid(x, y, z, indexing='ij'), dim=-1)
    return vox_coords

query_points = get_dense_pointcloud(num_steps).unsqueeze(0).to(device)
query_points = query_points.reshape(1,-1,3)

network = SDFNetwork(len(dataset_train), latent_size)
network.load_state_dict(torch.load("msia_8_model_multi.ckpt"))
network.to(device)
network.eval()

with torch.no_grad():
  shape_ids = torch.tensor([1], dtype=torch.long, device=device)
  occ_pred = network(shape_ids, query_points).squeeze(0)
  occ_pred = torch.sigmoid(occ_pred)
  occ_pred = occ_pred.reshape(num_steps, num_steps, num_steps)
  occ_pred = occ_pred.cpu().numpy()
  verts, faces, normals, values = measure.marching_cubes(occ_pred, 0.5)
  verts = verts/num_steps - 0.5

fig = go.Figure(go.Mesh3d(
        x=verts[:,0], y=verts[:,1], z=verts[:,2],
        i=faces[:,0], j=faces[:,1], k=faces[:,2],))
fig.show()

### Shape interpolation

One interesting experiment is to compute an interpolation between two shapes.

To do so:
* load the network
* cerate a network called `network_mix` with only one embedding. Load the weights for the pretrained model (except for embedding layer for which the size do not correspond)
* fill the embedding of `network_mix` with a weighted sum of the embedding of the two shapes.
* display the interpolated meshes

**Question:** implement shape interpolation

In [ ]:
if not os.path.exists("msia_8_model_multi.ckpt"):
  ! wget https://github.com/aboulch/MSIA_points/releases/download/v0.0.0/msia_8_model_multi.ckpt

from skimage import measure

num_steps = 32

def get_dense_pointcloud(num_steps):
    x = torch.arange(-0.5,0.5,1/num_steps)
    y = torch.arange(-0.5,0.5,1/num_steps)
    z = torch.arange(-0.5,0.5,1/num_steps)
    vox_coords = torch.stack(torch.meshgrid(x, y, z, indexing='ij'), dim=-1)
    return vox_coords

query_points = get_dense_pointcloud(num_steps).unsqueeze(0).to(device)
query_points = query_points.reshape(1,-1,3)

network = SDFNetwork(1272, latent_size)
ckpt = torch.load("msia_8_model_multi.ckpt")
network.load_state_dict(ckpt, strict=False)
network.to(device)
network.eval()

network_mix = SDFNetwork(1, latent_size)
ckpt = torch.load("msia_8_model_multi.ckpt")
ckpt.pop("shape_embedding.weight")
network_mix.load_state_dict(ckpt, strict=False)
network_mix.to(device)
network_mix.eval()

print(network_mix.shape_embedding.weight.shape)

meshes = []

for a_id, a in enumerate(torch.arange(0,1.1,0.1)):
  print(a)

  with torch.no_grad():
    shape_ids = torch.tensor([0], dtype=torch.long, device=device)
    network_mix.shape_embedding.weight[0] = a*network.shape_embedding.weight[0] + (1-a)*a*network.shape_embedding.weight[2]
    occ_pred = network_mix(shape_ids, query_points).squeeze(0)
    occ_pred = torch.sigmoid(occ_pred)
    occ_pred = occ_pred.reshape(num_steps, num_steps, num_steps)
    occ_pred = occ_pred.cpu().numpy()
    verts, faces, normals, values = measure.marching_cubes(occ_pred, 0.5)
    verts = verts/num_steps - 0.5
    verts[:,0] += a_id * 0.8

    meshes.append(
        go.Mesh3d(
        x=verts[:,0], y=verts[:,1], z=verts[:,2],
        i=faces[:,0], j=faces[:,1], k=faces[:,2],)
    )
fig = go.Figure(
    data=meshes,
        layout=dict(
            scene=dict(
                xaxis=dict(visible=False),
                yaxis=dict(visible=False),
                zaxis=dict(visible=False),
                aspectmode="data", #this string can be 'data', 'cube', 'auto', 'manual'
                #a custom aspectratio is defined as follows:
                aspectratio=dict(x=1, y=1, z=0.95)
            )
        ))
fig.show()

torch.Size([1, 512])
tensor(0.)
tensor(0.1000)
tensor(0.2000)
tensor(0.3000)
tensor(0.4000)
tensor(0.5000)
tensor(0.6000)
tensor(0.7000)
tensor(0.8000)
tensor(0.9000)
tensor(1.)


## To go further: the latent of a unknown shape

Given a pretrained model is possible to optimize the latent representation of a new shape, it is a similar setup as the single shape optimization, but with a frozen pretrained MLP. Only the embedding is learned.

**Optional question:** implement the new shape latent representation.